In [2]:
import pandas as pd

#Loading data
ACLED_df = pd.read_csv('../Spatial Joins/ACLED Join/joined_ACLED_ZIP_22_PA.csv')

In [5]:
#Function to parse and clean actor names
def actor_names(actor_string):

    #checking if input is string
    if isinstance(actor_string, str):

        #splitting string by semicolon + deleting whitespace to extract actor names 
        actors = [actor.strip() for actor in actor_string.split(';')]

        #splitting further by colon if possible to keep main name of actor 
        actors = [actor.split(':')[-1].strip() for actor in actors]
        return actors
    
    else: 

        #return empty list 
        return[]

In [10]:
#applying function + expanding list into separate rows so each actors becomes individual data point 
actor_list_expanded = ACLED_df['assoc_actor_1'].apply(actor_names).explode().reset_index(drop=True)

#grouping by actor names and counting occurence
actor_counts = actor_list_expanded.value_counts()


print(actor_counts)

assoc_actor_1
Labor Group (United States)                218
Government of the United States (2021-)    172
Democratic Party                           164
Students (United States)                   139
Women (United States)                      130
                                          ... 
Workers World Party                          1
IfNotNow                                     1
Private Security Forces (United States)      1
Everytown for Gun Safety                     1
Abolish ICE                                  1
Name: count, Length: 150, dtype: int64


In [15]:
#converting series to df + naming cols
actor_counts_df = actor_counts.reset_index()
actor_counts_df.columns = ['Actor', 'Count']

#converting to csv
actor_counts_df.to_csv("actor_counts.csv", index=False)

#### Creating columns for presence of actor types 

In [20]:
#Loading data
actor_actorytype_df = pd.read_excel('../feature space/ACLED_Actor_ActorType_mapping.xlsx')

In [17]:
#Joining count with mapping (actually just nice to have, counts not necessary for features)
#merged_actor_mapping = pd.merge(left=actor_counts_df, right=actor_actorytype, on=['Actor'], how='inner')

In [23]:
#Creating dictionary for mapping actor names to types
actor_type_map = actor_actorytype_df.set_index('Actor')['actor_type'].to_dict()

In [27]:
#Applying the function and get a list of types per row
ACLED_df['Actor_Types'] = ACLED_df['assoc_actor_1'].apply(actor_names).apply(lambda actors: [actor_type_map.get(actor) for actor in actors if actor in actor_type_map])

In [28]:
#Creating binary columns for each actor type (defined in actor_actorytype_df)
for type_id in sorted(actor_actorytype_df['actor_type'].unique()):
    ACLED_df[f'actor_type{type_id}'] = ACLED_df['Actor_Types'].apply(lambda types: 1 if type_id in types else 0)

In [29]:
#Saving as csv (fix logic/order)
ACLED_df.to_csv("joined_ACLED_ZIP_22_PA.csv", index=False)